# Évaluation des Dégâts Post-Catastrophe

Ce notebook utilise les anomalies de caractéristiques Prithvi pour identifier les zones impactées par des séismes ou cyclones.

In [ ]:
!pip install geemap earthengine-api rasterio terratorch torch matplotlib -q
import ee, geemap, torch, rasterio
import numpy as np
import matplotlib.pyplot as plt
from terratorch import BACKBONE_REGISTRY

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([14.7, -5.3, 14.9, -5.1])
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).median().clip(roi)
geemap.ee_export_image(image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']), 'input.tif', scale=30, region=roi)

In [ ]:
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('input.tif') as src: img = src.read().astype(np.float32) / 10000.0
with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
h_feat = int(np.sqrt(feats.shape[1]-1))
variation = np.var(feats[0, 1:].numpy(), axis=1).reshape(h_feat, -1)

plt.imshow(variation, cmap='inferno')
plt.title("Zones de Dégâts (Analyse de Variance Prithvi)")
plt.show()